# Experiment: Regression Model Evaluation Techniques

**Objective:** To explore and apply various evaluation techniques for regression models using Python and the Scikit-learn library.

## Learning outcomes
After completing this practical, students will be able to:
1. Load an existing dataset and separate input features from the target.
2. Split data into training and testing sets.
3. Train regression models using Scikit-learn.
4. Evaluate predictions using MAE, MSE, RMSE, R², and Adjusted R².
5. Visualize actual versus predicted values and residuals.
6. Understand why evaluation on unseen test data is important.

**Dataset:** Diabetes dataset included with Scikit-learn. It is loaded directly from the library, so no separate file download is required.

**Note:** Run the cells in order. The exact results are generated when the notebook runs.

## 1. Import the required libraries

We use:
- `pandas` to display data in table form.
- `matplotlib` to plot evaluation graphs.
- Scikit-learn to load data, split it, train models, and calculate metrics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Display plots inside the notebook
%matplotlib inline

print("Libraries imported successfully.")

## 2. Load an existing dataset

The Scikit-learn Diabetes dataset is a built-in regression dataset. The target is a quantitative measure of disease progression one year after baseline. It is a teaching dataset, not a tool for diagnosing an individual.

`X` contains the input features and `y` contains the numerical target we want to predict.

In [ ]:
# Load the built-in Diabetes dataset
diabetes = load_diabetes()

# Convert the feature matrix into a DataFrame for easier inspection
X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)

# Target values are continuous numerical values
y = pd.Series(diabetes.target, name="disease_progression")

print("Dataset shape:", X.shape)
print("Number of samples:", X.shape[0])
print("Number of input features:", X.shape[1])

# Display the first five rows of the input data
display(X.head())

# Display the first five target values
display(y.head())

## 3. Inspect the dataset

Before training a model, inspect the data types, missing values, and basic statistics. This helps identify potential data-quality problems.

In [ ]:
# Summary of data types and non-null values
X.info()

# Count missing values in each feature
print("\\nMissing values per feature:")
display(X.isnull().sum())

# Descriptive statistics for the features
display(X.describe().round(3))

## 4. Split data into training and testing sets

We keep 20% of the data for testing and use 80% for training. The model learns from the training set; the test set estimates how well it performs on unseen samples.

`random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 5. Train two regression models

We compare:
1. **Linear Regression** — learns a linear relationship between features and the target. Standardization is included in a pipeline.
2. **Random Forest Regressor** — combines predictions from multiple decision trees.

The pipeline ensures that the scaler is fitted on the training data only, helping prevent data leakage.

In [ ]:
# Linear Regression model with feature standardization
linear_model = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

# Random Forest model
forest_model = RandomForestRegressor(
    n_estimators=100,  # Number of decision trees
    random_state=42,
    min_samples_leaf=2
)

# Train both models using only the training data
linear_model.fit(X_train, y_train)
forest_model.fit(X_train, y_train)

print("Both regression models have been trained.")

## 6. Generate predictions

`predict()` uses the trained model to estimate target values for the test features. These predicted values are compared with the true test targets during evaluation.

In [ ]:
y_pred_linear = linear_model.predict(X_test)
y_pred_forest = forest_model.predict(X_test)

# Show a small sample of actual and predicted values
comparison = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Linear Regression Prediction": y_pred_linear,
    "Random Forest Prediction": y_pred_forest
})

display(comparison.head(10).round(2))

## 7. Evaluation metrics

### A. Mean Absolute Error (MAE)

MAE is the average absolute difference between actual and predicted values. Lower MAE is better.

\[
MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|
\]

### B. Mean Squared Error (MSE)

MSE averages the squared prediction errors. Because errors are squared, larger errors receive more weight. Lower MSE is better.

\[
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
\]

### C. Root Mean Squared Error (RMSE)

RMSE is the square root of MSE. It is expressed in the same units as the target. Lower RMSE is better.

\[
RMSE = \sqrt{MSE}
\]

### D. R-squared (R²)

R² compares the model with a baseline that predicts the mean target value. A value closer to 1 generally indicates a better fit. It can be negative when the model performs worse than this baseline on the evaluated data.

\[
R^2 = 1-\frac{\sum_i(y_i-\hat{y}_i)^2}{\sum_i(y_i-\bar{y})^2}
\]

### E. Adjusted R-squared

Adjusted R² accounts for the number of predictors and the number of observations. It can be useful when comparing models with different numbers of predictors, but it is not a replacement for test-set evaluation.

\[
R^2_{\mathrm{adj}}=1-(1-R^2)\frac{n-1}{n-p-1}
\]

Here, \(n\) is the number of test observations and \(p\) is the number of input features. The formula requires \(n > p+1\).

## 8. Calculate all metrics

We calculate the metrics on the test set, not on the training data. This gives a more useful estimate of performance on unseen samples.

For MAE, MSE, and RMSE, smaller values are better. For R² and Adjusted R², values closer to 1 are generally preferable, though R² may be negative.

In [ ]:
def calculate_regression_metrics(y_true, y_pred, number_of_features):
    # MAE: average absolute prediction error
    mae = mean_absolute_error(y_true, y_pred)

    # MSE: average squared prediction error
    mse = mean_squared_error(y_true, y_pred)

    # RMSE: square root of MSE; same unit as the target
    rmse = np.sqrt(mse)

    # R-squared: goodness of fit relative to predicting the mean
    r2 = r2_score(y_true, y_pred)

    # Adjusted R-squared; defined here because test samples > features + 1
    n = len(y_true)
    p = number_of_features
    adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
        "Adjusted R2": adjusted_r2
    }

linear_metrics = calculate_regression_metrics(
    y_test, y_pred_linear, X_test.shape[1]
)

forest_metrics = calculate_regression_metrics(
    y_test, y_pred_forest, X_test.shape[1]
)

results = pd.DataFrame({
    "Linear Regression": linear_metrics,
    "Random Forest": forest_metrics
}).T

display(results.round(3))

## 9. Interpret the results

Use the table produced above:
- Compare MAE, MSE, and RMSE: lower values indicate smaller errors.
- Compare R²: a higher value generally indicates that the predictions explain more variation in the test targets.
- Do not choose a model from one metric alone. Consider the purpose of the task and the pattern of errors.
- The model with the best training score may not be the best model on unseen data.

The scores depend on the dataset and the train/test split. Do not assume a particular score before running the notebook.

## 10. Actual versus predicted plot

Points close to the diagonal line represent predictions close to the actual values. Points far from the line represent larger prediction errors.

In [ ]:
# Plot predictions from both models against actual target values
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

plot_data = [
    ("Linear Regression", y_pred_linear),
    ("Random Forest", y_pred_forest)
]

# Use the same diagonal reference range for both plots
low = min(y_test.min(), y_pred_linear.min(), y_pred_forest.min())
high = max(y_test.max(), y_pred_linear.max(), y_pred_forest.max())

for ax, (model_name, predictions) in zip(axes, plot_data):
    ax.scatter(y_test, predictions, alpha=0.75)
    ax.plot([low, high], [low, high], linestyle="--")
    ax.set_xlabel("Actual values")
    ax.set_ylabel("Predicted values")
    ax.set_title(model_name)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Residual analysis

A residual is the difference between the actual value and the predicted value:

\[
Residual = Actual - Predicted
\]

Residuals near zero indicate small errors. A residual plot can reveal patterns, changing error spread, or unusually large errors. Ideally, residuals should be scattered around zero without a clear pattern.

In [ ]:
# Residuals for each model
residuals_linear = y_test.to_numpy() - y_pred_linear
residuals_forest = y_test.to_numpy() - y_pred_forest

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

residual_data = [
    ("Linear Regression", y_pred_linear, residuals_linear),
    ("Random Forest", y_pred_forest, residuals_forest)
]

for ax, (name, predictions, residuals) in zip(axes, residual_data):
    ax.scatter(predictions, residuals, alpha=0.75)
    ax.axhline(y=0, linestyle="--")
    ax.set_xlabel("Predicted values")
    ax.set_ylabel("Residual (actual - predicted)")
    ax.set_title(name + " Residual Plot")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Cross-validation for Linear Regression

A single train/test split can produce a score that depends on which samples were selected. Cross-validation evaluates the model across multiple folds.

The following code uses five-fold cross-validation and reports R² for each fold and the mean. The scaler is inside the pipeline, so it is refitted within each training fold.

In [ ]:
# Five-fold cross-validation on the full feature table
cv_scores = cross_val_score(
    linear_model,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R2 score for each fold:", np.round(cv_scores, 3))
print("Mean cross-validation R2:", round(cv_scores.mean(), 3))
print("Standard deviation:", round(cv_scores.std(), 3))

## 13. Optional: Mean Absolute Percentage Error (MAPE)

MAPE reports the average absolute error as a percentage. It can be difficult to interpret when true values are zero or very close to zero. Use it only when percentage errors make sense for the target.

The Diabetes target is positive in this dataset, so this example can be run, but MAPE still has limitations for low target values.

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

mape = mean_absolute_percentage_error(y_test, y_pred_linear) * 100
print(f"Linear Regression MAPE: {mape:.2f}%")

## 14. Conclusion

In this experiment, we loaded an existing dataset, split it into training and testing sets, trained Linear Regression and Random Forest regression models, and evaluated their predictions using MAE, MSE, RMSE, R², and Adjusted R². We also visualized actual versus predicted values, examined residuals, and used cross-validation to assess consistency.

**Key takeaway:** No single metric tells the whole story. Compare several metrics, inspect residual plots, and evaluate on data that was not used for training.

## 15. Practice questions

Answer any **3** questions:
1. What is the difference between MAE and MSE?
2. Why is RMSE expressed in the same unit as the target?
3. What does an R² score of 0.80 indicate?
4. Why should model performance be evaluated on test data?
5. What is a residual, and how is it calculated?
6. Why is cross-validation useful?
7. What are the limitations of MAPE?
8. Compare the Linear Regression and Random Forest results produced by your notebook.